# OpenMetadata AI Agent
### Built by Baibhav Prateek | OpenMetadata Hackathon 2026

## What is this?
This notebook builds an intelligent AI agent that automatically 
decides how to search OpenMetadata based on your question.

## How it works:
1) You ask a question in plain English
2) The AI agent decides which tool to use
3) It fetches the right data from OpenMetadata
4) It gives you a clear and  helpful answers

## Why I built this:
Most data catalog tools require you to know exactly what to search for
This agent makes it conversational ,just ask naturally!!!

In [1]:

import requests
import json
from groq import Groq

# Your credentials
GROQ_API_KEY = "your_groq_api_key_here"
BASE_URL = "https://sandbox.open-metadata.org"
TOKEN = "your_openmetadata_token_here"

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

client = Groq(api_key=GROQ_API_KEY)
print("✅ Agent setup complete!")

✅ Agent setup complete!


In [2]:
# These are the tools my agent can use

# Tool 1: Get a list of tables

def tool_get_tables(limit=10):
    """Get list of tables from OpenMetadata"""
    response = requests.get(
        f"{BASE_URL}/api/v1/tables",
        headers=HEADERS,
        params={"limit": limit}
    )
    tables = response.json().get("data", [])
    return [{"name": t.get("name"), 
             "description": t.get("description", "No description"),
             "columns": len(t.get("columns", []))} 
            for t in tables]

    
# Tool 2: Search for specific tables by keyword  


def tool_search_tables(keyword):
    """Search for specific tables by keyword"""
    response = requests.get(
        f"{BASE_URL}/api/v1/search/query",
        headers=HEADERS,
        params={"q": keyword, "index": "table_search_index", "limit": 5}
    )
    hits = response.json().get("hits", {}).get("hits", [])
    return [h.get("_source", {}).get("name", "") for h in hits]



# Tool 3: Get all databases

def tool_get_databases():
    """Get all databases"""
    response = requests.get(
        f"{BASE_URL}/api/v1/databases",
        headers=HEADERS,
        params={"limit": 10}
    )
    dbs = response.json().get("data", [])
    return [d.get("name") for d in dbs]

print("✅ Agent tools ready!")

✅ Agent tools ready!


In [3]:
# This is the brain of the agent
# First it thinks about which tool to use
# Then it uses that tool to fetch data
# Finally it gives a human-friendly answer
def run_agent(user_question):
    print(f"🧠 Agent thinking about: {user_question}")
    print("-" * 50)
    
    # Step 1: Ask AI to decide what to do
    decision_prompt = f"""You are a data catalog agent. 
You have these tools available:
1. get_tables - fetches list of tables
2. search_tables - searches tables by keyword
3. get_databases - fetches all databases

User question: {user_question}

Which tool should you use first? Reply with ONLY one of:
get_tables
search_tables: <keyword>
get_databases"""

    decision = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": decision_prompt}]
    ).choices[0].message.content.strip()
    
    print(f"🔧 Agent decided to use: {decision}")
    
    # Step 2: Execute the chosen tool
    if "search_tables" in decision:
        keyword = decision.split(":")[-1].strip()
        data = tool_search_tables(keyword)
        tool_used = f"search_tables('{keyword}')"
    elif "get_databases" in decision:
        data = tool_get_databases()
        tool_used = "get_databases()"
    else:
        data = tool_get_tables()
        tool_used = "get_tables()"
    
    print(f"📦 Data fetched: {data}")
    print("-" * 50)
    
    # Step 3: Ask AI to answer using the fetched data
    final_prompt = f"""You are a helpful data catalog assistant.
User asked: {user_question}
You used tool: {tool_used}
Data retrieved: {data}

Now answer the user's question clearly and helpfully."""

    final_answer = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}]
    ).choices[0].message.content

    print(f"🤖 Agent answer: {final_answer}")
    return final_answer

print("✅ Agent brain ready!")

✅ Agent brain ready!


In [4]:
# Test the agent with different questions
test_questions = [
    "Which databases contain customer information?",
    "Find all tables that might have sensitive financial data",
    "What tables should a new data analyst explore first?"
]

print("=" * 60)
print("   🤖 My OpenMetadata AI Agent")
print("   Built for OpenMetadata Hackathon 2026")
print("=" * 60)

for question in test_questions:
    print(f"\n❓ User: {question}")
    print("=" * 60)
    run_agent(question)
    print()

print("✅ Agent demo complete!")

   🤖 OpenMetadata AI Agent Demo

❓ User: Show me all available databases
🧠 Agent thinking about: Show me all available databases
--------------------------------------------------
🔧 Agent decided to use: get_databases
📦 Data fetched: ['ACMECORP_GOLD', 'acme_raw', 'ANALYTICS', 'ANALYTICS_DB', 'database_well_successful_86bb6110', 'default', 'default', 'default', 'demo-test-cat', 'demo-test-cat']
--------------------------------------------------
🤖 Agent answer: You have requested a list of all available databases. After retrieving the data, I found that there are several databases available. Here is the list:

1. ACMECORP_GOLD
2. acme_raw
3. ANALYTICS
4. ANALYTICS_DB
5. database_well_successful_86bb6110
6. default (please note that there are multiple instances of this database, it's possible that they are duplicates or have different purposes)
7. demo-test-cat (also appears to be duplicated)

Please let me know if you need more information about any of these databases or if there's anyth